# Evidencia 3 — Clasificador Vehicular CNN
## Detección de Flotas Logísticas mediante Clasificación Vehicular con CNN
**ISPC — Procesamiento de Imágenes 2026**  
Mario Arce │ Ariel Denaro │ Simón Vottero

## 0. Instalación de dependencias

In [ ]:
# Instalar scikit-learn para métricas avanzadas (ya incluido en Colab, pero por si acaso)
# !pip install scikit-learn --quiet

## 1. Importaciones

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import math
import os

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    roc_auc_score
)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponible: {tf.config.list_physical_devices('GPU')}")

## 2. Carga y filtrado del dataset

In [ ]:
datos, metadatos = tfds.load('cifar10', as_supervised=True, with_info=True)

CLASE_LIVIANO = 1  # automobile
CLASE_PESADO  = 9  # truck

nombres_clases = ['Liviano', 'Pesado']

print("Clases originales de CIFAR-10:")
print(metadatos.features['label'].names)
print(f"\nClases seleccionadas:")
print(f"  {CLASE_LIVIANO} → automobile → Liviano")
print(f"  {CLASE_PESADO}  → truck      → Pesado")

## 3. Filtrado y reasignación de etiquetas

In [ ]:
def filtrar_vehiculos(imagen, etiqueta):
    return tf.logical_or(
        tf.equal(etiqueta, CLASE_LIVIANO),
        tf.equal(etiqueta, CLASE_PESADO)
    )

def remap_etiquetas(imagen, etiqueta):
    nueva_etiqueta = tf.cast(tf.equal(etiqueta, CLASE_PESADO), tf.int64)
    return imagen, nueva_etiqueta

datos_entrenamiento_raw = datos['train'].filter(filtrar_vehiculos).map(remap_etiquetas)
datos_pruebas_raw       = datos['test'].filter(filtrar_vehiculos).map(remap_etiquetas)

print("Dataset filtrado: automobile (Liviano=0) vs truck (Pesado=1)")
print("Imágenes de entrenamiento: ~10.000 | Imágenes de prueba: ~2.000")

## 4. Normalización y pipeline de datos

In [ ]:
def normalizar(imagenes, etiquetas):
    """Normalización Min-Max: [0, 255] → [0.0, 1.0]"""
    imagenes = tf.cast(imagenes, tf.float32) / 255.0
    return imagenes, etiquetas

# -------------------------------------------------------
# Data Augmentation — solo para entrenamiento
# Las capas de augmentation se incluyen DENTRO del modelo
# para ejecutarse en GPU (paradigma moderno, Módulo 3)
# -------------------------------------------------------

EJ_ENTRENAMIENTO = 10000
EJ_PRUEBAS       = 2000
LOTE             = 32

datos_entrenamiento = (
    datos_entrenamiento_raw
    .map(normalizar)
    .cache()
    .shuffle(EJ_ENTRENAMIENTO)
    .batch(LOTE)
    .repeat()
)

datos_pruebas = (
    datos_pruebas_raw
    .map(normalizar)
    .cache()
    .batch(LOTE)
)

print("Pipeline de datos configurado.")

## 5. Visualización exploratoria del dataset

In [ ]:
plt.figure(figsize=(12, 12))
for i, (imagen, etiqueta) in enumerate(datos_entrenamiento_raw.take(25)):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(imagen.numpy())
    color = 'blue' if etiqueta.numpy() == 0 else 'red'
    plt.xlabel(nombres_clases[etiqueta.numpy()], color=color, fontsize=9)
plt.suptitle("Dataset filtrado — Azul: Liviano | Rojo: Pesado", fontsize=13)
plt.tight_layout()
plt.show()

## 6. Detección de bordes con operadores clásicos (Módulo 2)

Aplicamos Sobel y Canny sobre muestras del dataset para analizar las características morfológicas diferenciales entre clases Liviano y Pesado.
Estos análisis justifican la necesidad de una arquitectura CNN capaz de aprender invarianzas espaciales.

In [ ]:
import cv2

def aplicar_operadores_bordes(imagen_rgb):
    """Aplica Sobel y Canny a una imagen RGB (valores float [0,1])."""
    # Convertir a uint8 BGR para OpenCV
    img_uint8 = (imagen_rgb * 255).astype(np.uint8)
    img_gray  = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)

    # Operador Sobel (primera derivada)
    sobel_x = cv2.Sobel(img_gray, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(img_gray, cv2.CV_64F, 0, 1, ksize=3)
    sobel   = cv2.convertScaleAbs(cv2.addWeighted(
                  cv2.convertScaleAbs(sobel_x), 0.5,
                  cv2.convertScaleAbs(sobel_y), 0.5, 0))

    # Filtro Laplaciano del Gaussiano (LoG)
    img_blur  = cv2.GaussianBlur(img_gray, (3, 3), 0)
    laplaciano = cv2.convertScaleAbs(cv2.Laplacian(img_blur, cv2.CV_64F, ksize=3))

    # Algoritmo de Canny
    canny = cv2.Canny(img_blur, 50, 120)

    return img_gray, sobel, laplaciano, canny


# Tomar 2 Livianos y 2 Pesados
muestras_liviano = []
muestras_pesado  = []

for img, lbl in datos_entrenamiento_raw.take(200):
    if lbl.numpy() == 0 and len(muestras_liviano) < 2:
        muestras_liviano.append(img.numpy())
    elif lbl.numpy() == 1 and len(muestras_pesado) < 2:
        muestras_pesado.append(img.numpy())
    if len(muestras_liviano) == 2 and len(muestras_pesado) == 2:
        break

muestras = muestras_liviano + muestras_pesado
etiquetas_muestras = ['Liviano'] * 2 + ['Pesado'] * 2

titulos_ops = ['Original', 'Escala grises', 'Sobel', 'Laplaciano (LoG)', 'Canny']

fig, axes = plt.subplots(4, 5, figsize=(16, 13))
fig.suptitle('Análisis de Operadores de Detección de Bordes por Clase', fontsize=14, fontweight='bold')

for fila, (img, lbl) in enumerate(zip(muestras, etiquetas_muestras)):
    gray, sobel, lap, canny = aplicar_operadores_bordes(img / 255.0 if img.max() > 1 else img)
    resultados = [img, gray, sobel, lap, canny]
    for col, (res, tit) in enumerate(zip(resultados, titulos_ops)):
        ax = axes[fila][col]
        if col == 0:
            ax.imshow(res.astype(np.uint8) if img.max() > 1 else res)
            ax.set_ylabel(lbl, fontsize=10, fontweight='bold',
                          color='blue' if lbl == 'Liviano' else 'red')
        else:
            ax.imshow(res, cmap='gray')
        if fila == 0:
            ax.set_title(tit, fontsize=10)
        ax.axis('off')

plt.tight_layout()
plt.show()

print("Observación: Canny y Sobel revelan diferencias morfológicas entre siluetas de")
print("vehículos livianos (bordes curvos, perfil bajo) vs pesados (bordes angulares,")
print("cabina alta, caja de carga rectangular). Estas diferencias son las que la CNN")
print("aprenderá a detectar mediante sus filtros convolucionales entrenables.")

## 7. Arquitectura CNN

Reemplazamos el MLP baseline de la Evidencia 2 por una **Red Neuronal Convolucional** completa:

- **Data Augmentation** integrada en la red (RandomFlip, RandomRotation) → ejecuta en GPU
- **Bloques Conv2D + BatchNorm + MaxPooling** → extraen jerarquía de características espaciales
- **Dropout** → regularización para evitar overfitting
- **GlobalAveragePooling2D** → reemplaza Flatten+Dense pesado (como en ResNet, Módulo 4)
- **Salida sigmoide** → apropiada para clasificación binaria

In [ ]:
def construir_cnn():
    """
    CNN para clasificación binaria Liviano/Pesado sobre CIFAR-10 (32x32x3).

    Arquitectura:
      Augmentation → Block1(32f) → Block2(64f) → Block3(128f)
      → GlobalAvgPool → Dense(256) → Dropout → Dense(1, sigmoid)
    """
    modelo = tf.keras.Sequential([

        # --- Entrada ---
        tf.keras.layers.Input(shape=(32, 32, 3), dtype=tf.float32),

        # --- Data Augmentation (solo activo en entrenamiento) ---
        tf.keras.layers.RandomFlip('horizontal'),
        tf.keras.layers.RandomRotation(0.10),
        tf.keras.layers.RandomZoom(0.10),

        # --- Bloque Convolucional 1: detecta bordes y texturas simples ---
        tf.keras.layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D((2, 2)),          # 32x32 → 16x16
        tf.keras.layers.Dropout(0.25),

        # --- Bloque Convolucional 2: detecta formas y contornos ---
        tf.keras.layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D((2, 2)),          # 16x16 → 8x8
        tf.keras.layers.Dropout(0.25),

        # --- Bloque Convolucional 3: detecta estructuras de alto nivel ---
        tf.keras.layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D((2, 2)),          # 8x8 → 4x4
        tf.keras.layers.Dropout(0.40),

        # --- Cabeza de clasificación ---
        # GlobalAveragePooling2D: promedia cada mapa de características
        # a un escalar → vector 1D de 128 elementos sin Flatten pesado
        tf.keras.layers.GlobalAveragePooling2D(),

        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.50),

        # Salida sigmoide para clasificación binaria
        # P(Pesado) → umbral 0.5 → clase Liviano (0) o Pesado (1)
        tf.keras.layers.Dense(1, activation='sigmoid')
    ], name='CNN_Vehicular_v3')

    return modelo


modelo = construir_cnn()
modelo.summary()

## 8. Compilación

In [ ]:
modelo.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    # BinaryCrossentropy + sigmoid: apropiado para clasificación binaria
    # (SparseCategoricalCrossentropy era para la salida de 2 neuronas del MLP baseline)
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc')
    ]
)
print("Modelo compilado.")

## 9. Callbacks de entrenamiento

In [ ]:
callbacks = [
    # EarlyStopping: detiene el entrenamiento si val_loss no mejora en 8 épocas
    # y restaura los mejores pesos (Módulo 3 — Parada Temprana)
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    # ReduceLROnPlateau: reduce la tasa de aprendizaje si val_loss se estanca
    # Implementa el Learning Rate Scheduler mencionado en Módulo 3
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1
    ),
    # ModelCheckpoint: guarda el mejor modelo en disco
    tf.keras.callbacks.ModelCheckpoint(
        filepath='mejor_modelo_cnn_vehicular.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]
print("Callbacks configurados: EarlyStopping | ReduceLROnPlateau | ModelCheckpoint")

## 10. Split de validación

Separamos un 20% del conjunto de entrenamiento como conjunto de validación para monitorear overfitting durante el entrenamiento (OE3 — próximos pasos de E2).

In [ ]:
# Construir datasets con validación explícita
SPLIT_VAL       = 0.20  # 20% para validación
EJ_VAL          = int(EJ_ENTRENAMIENTO * SPLIT_VAL)   # 2000
EJ_TRAIN_REAL   = EJ_ENTRENAMIENTO - EJ_VAL           # 8000

dataset_completo = (
    datos_entrenamiento_raw
    .map(normalizar)
    .shuffle(EJ_ENTRENAMIENTO, seed=42)
)

datos_val   = dataset_completo.take(EJ_VAL).batch(LOTE).cache()
datos_train = dataset_completo.skip(EJ_VAL).batch(LOTE).cache().repeat()

STEPS_PER_EPOCH  = math.ceil(EJ_TRAIN_REAL / LOTE)   # 250
VALIDATION_STEPS = math.ceil(EJ_VAL / LOTE)           # 63

print(f"Entrenamiento: {EJ_TRAIN_REAL} imágenes | {STEPS_PER_EPOCH} pasos/época")
print(f"Validación:    {EJ_VAL} imágenes  | {VALIDATION_STEPS} pasos")
print(f"Prueba:        {EJ_PRUEBAS} imágenes")

## 11. Entrenamiento de la CNN

In [ ]:
EPOCAS_MAX = 50  # EarlyStopping detendrá antes si corresponde

historial = modelo.fit(
    datos_train,
    epochs=EPOCAS_MAX,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_data=datos_val,
    validation_steps=VALIDATION_STEPS,
    callbacks=callbacks,
    verbose=1
)

print(f"\nEntrenamiento finalizado en {len(historial.history['loss'])} épocas.")

## 12. Curvas de aprendizaje

In [ ]:
hist = historial.history
epocas = range(1, len(hist['loss']) + 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Curvas de Aprendizaje — CNN Vehicular', fontsize=14, fontweight='bold')

# Pérdida
axes[0][0].plot(epocas, hist['loss'],     label='Entrenamiento', color='royalblue')
axes[0][0].plot(epocas, hist['val_loss'], label='Validación',    color='tomato', linestyle='--')
axes[0][0].set_title('Pérdida (Loss)')
axes[0][0].set_xlabel('Época')
axes[0][0].set_ylabel('Binary Crossentropy')
axes[0][0].legend()
axes[0][0].grid(True, alpha=0.3)

# Accuracy
axes[0][1].plot(epocas, hist['accuracy'],     label='Entrenamiento', color='royalblue')
axes[0][1].plot(epocas, hist['val_accuracy'], label='Validación',    color='tomato', linestyle='--')
axes[0][1].set_title('Exactitud (Accuracy)')
axes[0][1].set_xlabel('Época')
axes[0][1].set_ylabel('Accuracy')
axes[0][1].legend()
axes[0][1].grid(True, alpha=0.3)

# AUC
axes[1][0].plot(epocas, hist['auc'],     label='Entrenamiento', color='forestgreen')
axes[1][0].plot(epocas, hist['val_auc'], label='Validación',    color='darkorange', linestyle='--')
axes[1][0].set_title('AUC (Área Bajo la Curva ROC)')
axes[1][0].set_xlabel('Época')
axes[1][0].set_ylabel('AUC')
axes[1][0].legend()
axes[1][0].grid(True, alpha=0.3)

# Learning Rate
if 'lr' in hist:
    axes[1][1].plot(epocas, hist['lr'], color='purple')
    axes[1][1].set_title('Tasa de Aprendizaje (ReduceLROnPlateau)')
    axes[1][1].set_xlabel('Época')
    axes[1][1].set_ylabel('Learning Rate')
    axes[1][1].set_yscale('log')
    axes[1][1].grid(True, alpha=0.3)
else:
    axes[1][1].axis('off')

plt.tight_layout()
plt.show()

## 13. Evaluación formal sobre el conjunto de prueba

In [ ]:
resultados = modelo.evaluate(datos_pruebas, verbose=1)
metricas_nombres = modelo.metrics_names

print("\n" + "="*50)
print("RESULTADOS EN CONJUNTO DE PRUEBA")
print("="*50)
for nombre, valor in zip(metricas_nombres, resultados):
    print(f"  {nombre:12s}: {valor:.4f}")

## 14. Métricas avanzadas: Matriz de Confusión, Precision, Recall, F1, AUC-ROC

In [ ]:
# Recolectar predicciones y etiquetas reales
y_pred_prob = []
y_real      = []

for imagenes_batch, etiquetas_batch in datos_pruebas:
    probs = modelo.predict(imagenes_batch, verbose=0)
    y_pred_prob.extend(probs.flatten())
    y_real.extend(etiquetas_batch.numpy())

y_pred_prob = np.array(y_pred_prob)
y_real      = np.array(y_real)
y_pred      = (y_pred_prob >= 0.5).astype(int)

print("Predicciones recolectadas:", len(y_pred), "imágenes")

In [ ]:
# --- Reporte de clasificación completo ---
print("\nREPORTE DE CLASIFICACIÓN (sklearn)")
print("="*50)
print(classification_report(y_real, y_pred, target_names=['Liviano', 'Pesado']))

# --- AUC-ROC ---
auc_score = roc_auc_score(y_real, y_pred_prob)
print(f"AUC-ROC: {auc_score:.4f}")

In [ ]:
# --- Matriz de Confusión ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusión
cm = confusion_matrix(y_real, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Liviano', 'Pesado'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Matriz de Confusión', fontsize=13, fontweight='bold')

# VP, VN, FP, FN
VN, FP, FN, VP = cm.ravel()
axes[0].set_xlabel(
    f'VP={VP}  VN={VN}  FP={FP}  FN={FN}',
    fontsize=10
)

# Curva ROC
fpr, tpr, _ = roc_curve(y_real, y_pred_prob)
axes[1].plot(fpr, tpr, color='royalblue', lw=2,
             label=f'CNN (AUC = {auc_score:.3f})')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1, label='Azar (AUC = 0.5)')
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.02])
axes[1].set_xlabel('Tasa de Falsos Positivos')
axes[1].set_ylabel('Tasa de Verdaderos Positivos')
axes[1].set_title('Curva ROC', fontsize=13, fontweight='bold')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 15. KPI Logístico: Proporción de vehículos Pesados en la flota

In [ ]:
# Aplicar el modelo sobre el set de prueba completo y calcular
# la proporción de vehículos clasificados como Pesados
total           = len(y_pred)
n_pesados       = np.sum(y_pred == 1)
n_livianos      = np.sum(y_pred == 0)
prop_pesados    = n_pesados / total
prop_livianos   = n_livianos / total

# Ground truth
n_pesados_real  = np.sum(y_real == 1)
n_livianos_real = np.sum(y_real == 0)

print("=" * 50)
print("KPI LOGÍSTICO — COMPOSICIÓN DE FLOTA")
print("=" * 50)
print(f"\n{'Clase':<12} {'Real':>8} {'Predicho':>10} {'% Predicho':>12}")
print("-" * 44)
print(f"{'Liviano':<12} {n_livianos_real:>8} {n_livianos:>10} {prop_livianos*100:>11.1f}%")
print(f"{'Pesado':<12} {n_pesados_real:>8} {n_pesados:>10} {prop_pesados*100:>11.1f}%")
print(f"{'TOTAL':<12} {total:>8} {total:>10}")

# Gráfico de torta
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

labels = ['Liviano', 'Pesado']
colors = ['#4472C4', '#ED7D31']

axes[0].pie([n_livianos_real, n_pesados_real], labels=labels, colors=colors,
            autopct='%1.1f%%', startangle=90)
axes[0].set_title('Composición Real de la Flota')

axes[1].pie([n_livianos, n_pesados], labels=labels, colors=colors,
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Composición Detectada por el Modelo')

plt.suptitle('KPI Logístico — Proporción de Vehículos Pesados', fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n→ El modelo detecta un {prop_pesados*100:.1f}% de vehículos Pesados en la flota analizada.")
print(f"  La proporción real es {n_pesados_real/total*100:.1f}%.")
print(f"  Desviación del KPI: {abs(prop_pesados - n_pesados_real/total)*100:.2f} puntos porcentuales.")

## 16. Visualización de predicciones individuales con barras de confianza

In [ ]:
def graficar_predicciones(imagenes, etiquetas_reales, predicciones_prob, n=12):
    """Grilla de n imágenes con panel de confianza.
    Azul = predicción correcta | Rojo = predicción incorrecta
    """
    filas = n // 4
    fig = plt.figure(figsize=(16, filas * 3.5))
    fig.suptitle('Predicciones CNN — Azul: Correcto | Rojo: Incorrecto', fontsize=13)

    for i in range(n):
        prob_pesado  = float(predicciones_prob[i])
        prob_liviano = 1.0 - prob_pesado
        pred_clase   = 1 if prob_pesado >= 0.5 else 0
        real_clase   = int(etiquetas_reales[i])
        correcto     = pred_clase == real_clase
        color        = 'blue' if correcto else 'red'

        # Imagen
        ax_img = fig.add_subplot(filas, 8, i * 2 + 1)
        ax_img.imshow(imagenes[i])
        ax_img.set_title(
            f"Real: {nombres_clases[real_clase]}\nPred: {nombres_clases[pred_clase]}",
            color=color, fontsize=8
        )
        ax_img.axis('off')

        # Barra de confianza
        ax_bar = fig.add_subplot(filas, 8, i * 2 + 2)
        ax_bar.barh([0, 1], [prob_liviano, prob_pesado],
                   color=['#4472C4', '#ED7D31'])
        ax_bar.set_yticks([0, 1])
        ax_bar.set_yticklabels(['Liv', 'Pes'], fontsize=8)
        ax_bar.set_xlim([0, 1])
        ax_bar.set_xticks([0, 0.5, 1])
        ax_bar.tick_params(axis='x', labelsize=7)
        ax_bar.axvline(0.5, color='gray', linestyle='--', lw=0.8)

    plt.tight_layout()
    plt.show()


# Tomar un lote del conjunto de prueba
for imagenes_vis, etiquetas_vis in datos_pruebas.take(1):
    imagenes_vis  = imagenes_vis.numpy()
    etiquetas_vis = etiquetas_vis.numpy()

probs_vis = modelo.predict(imagenes_vis, verbose=0).flatten()
graficar_predicciones(imagenes_vis[:12], etiquetas_vis[:12], probs_vis[:12], n=12)

## 17. Inferencia sobre imagen individual (pipeline end-to-end)

In [ ]:
def inferencia_imagen_individual(imagen_norm, etiqueta_real):
    """
    Flujo completo de inferencia sobre una imagen aislada.
    Demuestra la corrección de la Batch Dimension (Módulo 3).
    """
    # Añadir dimensión de lote: (32, 32, 3) → (1, 32, 32, 3)
    img_batch = np.expand_dims(imagen_norm, axis=0)

    # Predicción
    prob_pesado  = float(modelo.predict(img_batch, verbose=0)[0][0])
    prob_liviano = 1.0 - prob_pesado
    pred_clase   = 1 if prob_pesado >= 0.5 else 0

    # Visualización
    fig, axes = plt.subplots(1, 2, figsize=(7, 3))

    axes[0].imshow(imagen_norm)
    correcto = pred_clase == int(etiqueta_real)
    color = 'green' if correcto else 'red'
    axes[0].set_title(
        f"Real: {nombres_clases[int(etiqueta_real)]}\n"
        f"Pred: {nombres_clases[pred_clase]}",
        color=color, fontsize=11
    )
    axes[0].axis('off')

    bars = axes[1].bar(
        nombres_clases,
        [prob_liviano * 100, prob_pesado * 100],
        color=['#4472C4', '#ED7D31']
    )
    axes[1].set_ylabel('Probabilidad (%)')
    axes[1].set_ylim([0, 110])
    axes[1].set_title('Confianza del modelo')
    for bar, val in zip(bars, [prob_liviano * 100, prob_pesado * 100]):
        axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
                    f'{val:.1f}%', ha='center', va='bottom', fontsize=11)

    plt.tight_layout()
    plt.show()

    print(f"Liviano:  {prob_liviano*100:.1f}%")
    print(f"Pesado:   {prob_pesado*100:.1f}%")
    print(f"Predicción: {nombres_clases[pred_clase]} | Real: {nombres_clases[int(etiqueta_real)]}")


# Ejecutar sobre la imagen índice 5
idx = 5
inferencia_imagen_individual(imagenes_vis[idx], etiquetas_vis[idx])

## 18. Comparación baseline MLP vs CNN

In [ ]:
# Resultados del MLP baseline (E2) — valores de referencia documentados
# En E2 se esperaba 65-75% de accuracy según el análisis del informe

accuracy_cnn = resultados[metricas_nombres.index('accuracy')] * 100

categorias   = ['Accuracy', 'AUC-ROC']
mlp_vals     = [70.0,  0.72]   # Valores representativos del baseline E2
cnn_vals     = [accuracy_cnn, auc_score * 100]

x = np.arange(len(categorias))
ancho = 0.30

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - ancho/2, mlp_vals, ancho, label='MLP Baseline (E2)', color='#9E9E9E')
bars2 = ax.bar(x + ancho/2, cnn_vals,  ancho, label='CNN (E3)',          color='#4472C4')

ax.set_ylabel('Valor (%)')
ax.set_title('Comparación MLP Baseline (E2) vs CNN (E3)', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(categorias)
ax.set_ylim([0, 105])
ax.legend()
ax.grid(axis='y', alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():.1f}%', ha='center', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():.1f}%', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

mejora = accuracy_cnn - 70.0
print(f"Mejora de accuracy: +{mejora:.1f} puntos porcentuales sobre el MLP baseline.")

## 19. Guardar modelo final

In [ ]:
modelo.save('ev3_cnn_vehicular_final.keras')
print("Modelo guardado en: ev3_cnn_vehicular_final.keras")
print("\nArchivos generados:")
print("  ev3_cnn_vehicular_final.keras   — modelo completo")
print("  mejor_modelo_cnn_vehicular.keras — mejor checkpoint por val_accuracy")